In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

# Preparation of Data Frames

In [ ]:
df  = pd.DataFrame()

## Sentiment

In [ ]:
# Reading the saved data pickle file

df_sentiment  = pd.DataFrame()
df_time  = pd.DataFrame()
df_hour  = pd.DataFrame()

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentHourly.pkl', compression=None)
# df_sentiment["sentiment"] = df_sentiment["pos"]*100
# df_sentiment.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
# df_sentiment.index = pd.to_datetime(df_sentiment.Date)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)
# df_sentiment["sentiment"].ffill(inplace=True)

# df_sentiment["sentiment_yesterday"] = df_sentiment["sentiment"].shift(periods=1)
# df_sentiment["sentiment_day_before_yesterday"] = df_sentiment["sentiment"].shift(periods=2)

# # SENTIMENT RETURN
# df_sentiment["sentiment_return"]=(df_sentiment["sentiment_yesterday"]-df_sentiment["sentiment_day_before_yesterday"])/df_sentiment["sentiment_day_before_yesterday"]
# df_sentiment["sentiment_return"].ffill(inplace=True)

# # SENTIMENT LN RETURN
# df_sentiment["sentiment_ln_return"]=np.log(df_sentiment["sentiment_yesterday"]/df_sentiment["sentiment_day_before_yesterday"])
# df_sentiment["sentiment_ln_return"].ffill(inplace=True)

# # SENTIMENT EWM
# df_sentiment["sentiment_ewm"]=df_sentiment["sentiment"].ewm(10).mean()
# df_sentiment["sentiment_ewm"].ffill(inplace=True)

# # SENTIMENT EWM YESTERDAY
# df_sentiment["sentiment_yesterday_ewm"]=df_sentiment["sentiment_yesterday"].ewm(10).mean()
# df_sentiment["sentiment_yesterday_ewm"].ffill(inplace=True)


In [ ]:
df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment"].shift(periods=1)
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment"].shift(periods=2)
df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment(t-1)"].ffill()
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment(t-2)"].ffill()

# SENTIMENT RETURN
df_sentiment["sentiment_return"]=(df_sentiment["sentiment(t-1)"]-df_sentiment["sentiment(t-2)"])/df_sentiment["sentiment(t-2)"]
# SENTIMENT LN RETURN
df_sentiment["sentiment_ln_return"]=np.log(df_sentiment["sentiment(t-1)"]/df_sentiment["sentiment(t-2)"])
# SENTIMENT EWM
df_sentiment["sentiment_ewm10"]=df_sentiment["sentiment"].ewm(10).mean()
df_sentiment["sentiment_ewm20"]=df_sentiment["sentiment"].ewm(20).mean()

# POSITIVE LN RETURN
df_sentiment["pos(t-1)"] = df_sentiment["pos"].shift(periods=1)
df_sentiment["pos(t-2)"] = df_sentiment["pos"].shift(periods=2)
df_sentiment["pos(t-1)"] = df_sentiment["pos(t-1)"].ffill()
df_sentiment["pos(t-2)"] = df_sentiment["pos(t-2)"].ffill()
df_sentiment["pos_ln_return"]=np.log(df_sentiment["pos(t-1)"]/df_sentiment["pos(t-2)"])

# NEGATIVE LN RETURN
df_sentiment["neg(t-1)"] = df_sentiment["neg"].shift(periods=1)
df_sentiment["neg(t-2)"] = df_sentiment["neg"].shift(periods=2)
df_sentiment["neg(t-1)"] = df_sentiment["neg(t-1)"].ffill()
df_sentiment["neg(t-2)"] = df_sentiment["neg(t-2)"].ffill()
df_sentiment["neg_ln_return"]=np.log(df_sentiment["neg(t-1)"]/df_sentiment["neg(t-2)"])

# NEUTRAL LN RETURN
df_sentiment["neu(t-1)"] = df_sentiment["neu"].shift(periods=1)
df_sentiment["neu(t-2)"] = df_sentiment["neu"].shift(periods=2)
df_sentiment["neu(t-1)"] = df_sentiment["neu(t-1)"].ffill()
df_sentiment["neu(t-2)"] = df_sentiment["neu(t-2)"].ffill()
df_sentiment["neu_ln_return"]=np.log(df_sentiment["neu(t-1)"]/df_sentiment["neu(t-2)"])

# COMPOUND LN RETURN
df_sentiment["compound(t-1)"] = df_sentiment["compound"].shift(periods=1)
df_sentiment["compound(t-2)"] = df_sentiment["compound"].shift(periods=2)
df_sentiment["compound(t-1)"] = df_sentiment["compound(t-1)"].ffill()
df_sentiment["compound(t-2)"] = df_sentiment["compound(t-2)"].ffill()
df_sentiment["compound_ln_return"]=np.log(df_sentiment["compound(t-1)"]/df_sentiment["compound(t-2)"])

# PRESS RELEASE COUNT
df_sentiment["PressReleaseCount(t-1)"] = df_sentiment["PressReleaseCount"].shift(periods=1)
df_sentiment["PressReleaseCount(t-2)"] = df_sentiment["PressReleaseCount"].shift(periods=2)
df_sentiment["PressReleaseCount(t-1)"] = df_sentiment["PressReleaseCount(t-1)"].ffill()
df_sentiment["PressReleaseCount(t-2)"] = df_sentiment["PressReleaseCount(t-2)"].ffill()
df_sentiment["PressReleaseCount_ln_return"]=np.log(df_sentiment["PressReleaseCount(t-1)"]/df_sentiment["PressReleaseCount(t-2)"])
df_sentiment = df_sentiment.dropna()
# df_sentiment = df_sentiment.ffill()

In [ ]:
df_sentiment.loc[df_sentiment["Date"]=="2020-06-01"]

In [ ]:
df = df_sentiment

## VIX

In [ ]:
# from yahoo_fin.stock_info import get_data

# df_vix = pd.DataFrame(data=get_data("^VIX", start_date="25/09/2019", end_date="09/29/2021", index_as_date = True, interval="1d"))
# df_vix.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl")

In [ ]:
df_vix=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl', compression=None)

df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
df_vix.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_vix["vix"]  = df_vix["vix"].ffill()

df_vix["vix_yesterday"] = df_vix["vix"].shift(periods=1)
df_vix["vix_yesterday"].ffill(inplace=True)

# vix RETURN
df_vix["vix_return"]=(df_vix["vix"]-df_vix["vix_yesterday"])/df_vix["vix_yesterday"]
df_vix["vix_return"].ffill(inplace=True)

# vix LN RETURN
df_vix["vix_ln_return"]=np.log(df_vix["vix"]/df_vix["vix_yesterday"])
df_vix["vix_ln_return"].ffill(inplace=True)

# vix EWM
df_vix["vix_ewm"]=df_vix["vix"].ewm(10).mean()
df_vix["vix_ewm"].ffill(inplace=True)


In [ ]:
df_vix

In [ ]:
df.Date

In [ ]:
df

In [ ]:
df_vix

In [ ]:
df = df_vix.merge(df,how='left',right_on="Date", left_on="Date")

## S&P500

In [ ]:
# from yahoo_fin.stock_info import get_data

# df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="09/29/2021", index_as_date = True, interval="1d")
# df_sp500.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl")

In [ ]:
df_sp500=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl', compression=None)


df_sp500.index = pd.to_datetime(df_sp500.index)
df_sp500["Date"]= pd.to_datetime(df_sp500.index)

df_sp500["sp500"] = df_sp500["adjclose"]
df_sp500.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_sp500["sp500"]  = df_sp500["sp500"].ffill()

df_sp500["sp500_yesterday"] = df_sp500["sp500"].shift(periods=1)
# sp500 RETURN
df_sp500["sp500_return"]=(df_sp500["sp500"]-df_sp500["sp500_yesterday"])/df_sp500["sp500_yesterday"]
# sp500 LN RETURN
df_sp500["sp500_ln_return"]=np.log(df_sp500["sp500"]/df_sp500["sp500_yesterday"])
# sp500 EWM
df_sp500["sp500_ewm"]=df_sp500["sp500"].ewm(10).mean()

In [ ]:
df_sp500

In [ ]:
df = pd.merge(df_sp500,df,how='outer', on='Date')

In [ ]:
# df = df.drop([475])

# DAY MONTH YEAR

In [ ]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]

# MERGE

Missing Values, how to handle

Option 1: Fill NaN with Outlier or Zero
In this specific example filling the missing value with an outlier value such as np.inf or 0 seems to be very naive. However, using values like -999, is sometimes a good idea.

Option 2: Fill NaN with Mean Value
Filling NaNs with the mean value is also not sufficient and naive, and doesn't seems to be a good option.

Option 3: Fill NaN with Last Value with .ffill()
Filling NaNs with the last value could be bit better.

Option 4: Fill NaN with Linearly ffilld Value with .ffill()
Filling NaNs with the ffilld values is the best option in this small examlple but it requires knowledge of the neighouring value

# THREE PERIODS

In [ ]:
df.index = df.Date

In [ ]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':'2021-09-28']

In [ ]:
# df_part3 = df_part3.ffill()

In [ ]:
# df.reset_index(drop=True, inplace=True)
# df_part1.reset_index(drop=True, inplace=True)
# df_part2.reset_index(drop=True, inplace=True)
# df_part3.reset_index(drop=True, inplace=True)

In [ ]:
# df.sort_index()

In [ ]:
# df_part3['Num_word'] = df_part3['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text

In [ ]:
dfHour = pd.DataFrame()
dfHour.index = df_part3.loc[df_part3["Hour"]==0].Date

for i in (range(24)):

    dfHour["Hour"+str(i)+"_sentiment_ln_return"]=df_part3.loc[df_part3["Hour"]==i].sentiment_ln_return
    dfHour["Hour"+str(i)+"_sentiment_ln_return"].ffill(inplace=True)
    # maxnum = max(dfHour["Hour"+str(i)+"_sentiment"])
    # minnum =min(dfHour["Hour"+str(i)+"_sentiment"])
    # dfHour["Hour"+str(i)+"_sentiment"] = dfHour["Hour"+str(i)+"_sentiment"].replace(np.inf,maxnum+1)
    # dfHour["Hour"+str(i)+"_sentiment"] = dfHour["Hour"+str(i)+"_sentiment"].replace(-np.inf,minnum-1)
    dfHour["Hour"+str(i)+"_vix_ln_return"]=df_part3.loc[df_part3["Hour"]==i].vix_ln_return
    dfHour["Hour"+str(i)+"_vix_ln_return"].ffill(inplace=True)

df_part3['sentiment_ln_return'].dropna()
mx = df_part3.loc[df_part3['sentiment_ln_return'] != np.inf, 'sentiment_ln_return'].max()
df_part3['sentiment_ln_return'].replace(np.inf,mx,inplace=True)
mn = df_part3.loc[df_part3['sentiment_ln_return'] != -np.inf, 'sentiment_ln_return'].min()
df_part3['sentiment_ln_return'].replace(-np.inf,mn,inplace=True)

# df_part3.ffill(inplace=True)


In [ ]:
mx = df_part3.loc[df_part3['pos_ln_return'] != np.inf, 'pos_ln_return'].max()
df_part3['pos_ln_return'].replace(np.inf,0,inplace=True)
mn = df_part3.loc[df_part3['pos_ln_return'] != -np.inf, 'pos_ln_return'].min()
df_part3['pos_ln_return'].replace(-np.inf,0,inplace=True)

mx = df_part3.loc[df_part3['neg_ln_return'] != np.inf, 'pos_ln_return'].max()
df_part3['neg_ln_return'].replace(np.inf,0,inplace=True)
mn = df_part3.loc[df_part3['neg_ln_return'] != -np.inf, 'pos_ln_return'].min()
df_part3['neg_ln_return'].replace(-np.inf,0,inplace=True)

mx = df_part3.loc[df_part3['neu_ln_return'] != np.inf, 'neu_ln_return'].max()
df_part3['neu_ln_return'].replace(np.inf,0,inplace=True)
mn = df_part3.loc[df_part3['neu_ln_return'] != -np.inf, 'neu_ln_return'].min()
df_part3['neu_ln_return'].replace(-np.inf,0,inplace=True)

mx = df_part3.loc[df_part3['compound_ln_return'] != np.inf, 'compound_ln_return'].max()
df_part3['compound_ln_return'].replace(np.inf,0,inplace=True)
mn = df_part3.loc[df_part3['compound_ln_return'] != -np.inf, 'compound_ln_return'].min()
df_part3['compound_ln_return'].replace(-np.inf,0,inplace=True)